# E1 Compression Breakout — Parameter Optimization (Optuna)

**TRAIN WINDOW ONLY: 2023-01-01 → 2024-06-30**

Locked boundaries — never change these dates:
- Train (optimize here): 2023-01-01 → 2024-06-30
- Validation (locked reference, Sharpe 1.29): 2024-07-01 → 2025-01-01
- Out-of-sample (DO NOT TOUCH): 2025-01-01 → today

What we're optimizing:
- ATR compression threshold (how tight the squeeze must be)
- Range period (how many candles define the range high/low)
- Volume floor multiplier (how much volume surge is required)
- Stop loss / take profit percentages

What we're NOT optimizing (locked by spec):
- ATR period (14) — standard, changing it is overfitting
- EMA risk-off period (20) and threshold (0.97) — hard filter, not a signal param
- Time limit (24h) — operational constraint

In [ ]:
import sys, warnings, datetime
warnings.filterwarnings("ignore")
sys.path.insert(0, '/Users/hermes/quants-lab')

from decimal import Decimal
from app.controllers.directional_trading.e1_compression_breakout import E1CompressionBreakoutConfig
from core.backtesting.optimizer import BacktestingConfig, BaseStrategyConfigGenerator, StrategyOptimizer
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
from hummingbot.core.data_type.common import OrderType

# ── HARD-CODED TRAIN WINDOW ──────────────────────────────────────────────────
# DO NOT change these dates. Validation and OOS windows must stay unseen.
TRAIN_START = datetime.datetime(2023, 1, 1)
TRAIN_END   = datetime.datetime(2024, 6, 30)
# ─────────────────────────────────────────────────────────────────────────────

class E1ConfigGenerator(BaseStrategyConfigGenerator):
    """
    Optuna config generator for E1 Compression Breakout.
    Only the train window is ever passed to generate_config.
    """

    async def generate_config(self, trial) -> BacktestingConfig:
        # ── Params to optimize ───────────────────────────────────────────────
        atr_compression_threshold = trial.suggest_float(
            "atr_compression_threshold", 0.10, 0.35, step=0.05)
        range_period = trial.suggest_int(
            "range_period", 10, 40, step=5)
        volume_floor_multiplier = trial.suggest_float(
            "volume_floor_multiplier", 1.1, 2.0, step=0.1)
        stop_loss = trial.suggest_float(
            "stop_loss", 0.01, 0.025, step=0.005)
        take_profit = trial.suggest_float(
            "take_profit", 0.02, 0.05, step=0.005)
        trailing_activation = trial.suggest_float(
            "trailing_activation", 0.01, 0.025, step=0.005)
        trailing_delta = trial.suggest_float(
            "trailing_delta", 0.003, 0.01, step=0.001)

        # ── Locked params (do not suggest these) ────────────────────────────
        config = E1CompressionBreakoutConfig(
            id=f"e1_opt_{trial.number}",
            connector_name="binance_perpetual",
            trading_pair="BTC-USDT",
            total_amount_quote=Decimal("1000"),
            max_executors_per_side=1,
            cooldown_time=300,
            leverage=1,
            # Signal params — some optimized, some locked
            atr_period=14,                            # locked
            atr_compression_window=90  # unit: days,                # locked
            atr_compression_threshold=atr_compression_threshold,
            range_period=range_period,
            volume_period=20,                         # locked
            volume_floor_multiplier=volume_floor_multiplier,
            volume_spike_multiplier=2.0,              # locked
            risk_off_ema_period=20,                   # locked — hard filter
            risk_off_threshold=0.97,                  # locked — hard filter
            # Triple barrier
            stop_loss=Decimal(str(stop_loss)),
            take_profit=Decimal(str(take_profit)),
            time_limit=86400,                         # locked
            trailing_stop=TrailingStop(
                activation_price=Decimal(str(trailing_activation)),
                trailing_delta=Decimal(str(trailing_delta)),
            ),
            take_profit_order_type=OrderType.MARKET,
        )

        return BacktestingConfig(
            config=config,
            start=self.start,
            end=self.end,
        )

print("E1ConfigGenerator ready")
print(f"Train window: {TRAIN_START.date()} → {TRAIN_END.date()}")

## Run Optimization

`n_trials=100` is a good starting point — takes ~10-20 min depending on window size.
The objective maximizes Sharpe ratio on the train window only.

In [ ]:
config_generator = E1ConfigGenerator(start_date=TRAIN_START, end_date=TRAIN_END)
optimizer = StrategyOptimizer(load_cached_data=True)

await optimizer.optimize(
    study_name="e1_compression_breakout_train",
    config_generator=config_generator,
    n_trials=100,
)
print("Optimization complete")

## Optuna Dashboard

Launch the interactive dashboard to explore trial results, param importances, and Pareto fronts.

In [ ]:
optimizer.launch_optuna_dashboard()

## Analyze Results

Find stable params — not just the single best trial.
A good param set performs well across many trials, not just one lucky run.

In [ ]:
import pandas as pd
import plotly.express as px

study = optimizer.study
trials_df = study.trials_dataframe()

# Filter completed trials with valid Sharpe
completed = trials_df[trials_df["state"] == "COMPLETE"].copy()
completed = completed[completed["value"].notna() & (completed["value"] > 0)]
completed = completed.sort_values("value", ascending=False)

print(f"Completed trials: {len(completed)} / {len(trials_df)}")
print(f"\nTop 10 trials by Sharpe:")
cols = ["value"] + [c for c in completed.columns if c.startswith("params_")]
print(completed[cols].head(10).to_string())

In [ ]:
# Param importance — which params actually matter?
import optuna
importances = optuna.importance.get_param_importances(study)
imp_df = pd.DataFrame({"param": list(importances.keys()), "importance": list(importances.values())})
imp_df = imp_df.sort_values("importance", ascending=True)

fig = px.bar(imp_df, x="importance", y="param", orientation="h",
             title="Parameter Importance (Sharpe)", color="importance",
             color_continuous_scale="Viridis")
fig.update_layout(plot_bgcolor="rgba(0,0,0,0.85)", paper_bgcolor="rgba(0,0,0,0.85)",
                  font=dict(color="white"), showlegend=False)
fig.show()

In [ ]:
# Stability check: top params by median Sharpe across a neighbourhood
# Group by rounded param values, find clusters with consistently high Sharpe
param_cols = [c for c in completed.columns if c.startswith("params_")]

# Show distribution of Sharpe for top-10% of trials
top_decile = completed.head(max(10, len(completed) // 10))
print("Top decile Sharpe stats:")
print(top_decile["value"].describe().round(3))
print("\nTop decile param ranges:")
for col in param_cols:
    name = col.replace("params_", "")
    print(f"  {name}: {top_decile[col].min():.4f} – {top_decile[col].max():.4f}  (median {top_decile[col].median():.4f})")

## Select Best Params

Pick the param set from the stable region above (median of top decile, not single best trial).
Copy them into the validation backtest cell below.

In [ ]:
# Get best trial params
best = study.best_trial
print("Best trial Sharpe:", round(best.value, 4))
print("\nBest params:")
for k, v in best.params.items():
    print(f"  {k}: {v}")

## Validate on Held-Out Window

Run the optimized params on the **validation window only** (2024-07-01 → 2025-01-01).
This is a one-shot test — do not re-optimize based on this result.

Compare to baseline Sharpe 1.29 (unoptimized params on same window).
If optimized params score much higher, be suspicious of overfitting.

In [ ]:
from core.backtesting import BacktestingEngine

# ── VALIDATION WINDOW — ONE SHOT, DO NOT RE-OPTIMIZE ────────────────────────
VAL_START = datetime.datetime(2024, 7, 1)
VAL_END   = datetime.datetime(2025, 1, 1)
# ─────────────────────────────────────────────────────────────────────────────

# Paste best params here (from stability analysis above, not just best trial)
best_params = best.params  # or manually override with stable region median

val_config = E1CompressionBreakoutConfig(
    id="e1_validation",
    connector_name="binance_perpetual",
    trading_pair="BTC-USDT",
    total_amount_quote=Decimal("1000"),
    max_executors_per_side=1,
    cooldown_time=300,
    leverage=1,
    atr_period=14,
    atr_compression_window=90  # unit: days,
    atr_compression_threshold=best_params["atr_compression_threshold"],
    range_period=best_params["range_period"],
    volume_period=20,
    volume_floor_multiplier=best_params["volume_floor_multiplier"],
    volume_spike_multiplier=2.0,
    risk_off_ema_period=20,
    risk_off_threshold=0.97,
    stop_loss=Decimal(str(best_params["stop_loss"])),
    take_profit=Decimal(str(best_params["take_profit"])),
    time_limit=86400,
    trailing_stop=TrailingStop(
        activation_price=Decimal(str(best_params["trailing_activation"])),
        trailing_delta=Decimal(str(best_params["trailing_delta"])),
    ),
    take_profit_order_type=OrderType.MARKET,
)

bt = BacktestingEngine(load_cached_data=True)
val_result = await bt.run_backtesting(
    config=val_config,
    start=int(VAL_START.timestamp()),
    end=int(VAL_END.timestamp()),
    backtesting_resolution="1m",
    trade_cost=0.000375,
)

if not isinstance(val_result.results.get("close_types"), dict):
    val_result.results["close_types"] = {}

print("=== VALIDATION RESULT (optimized params) ===")
print(val_result.get_results_summary())
print(f"\nBaseline (unoptimized): Sharpe 1.29, adj PnL $76")
print(f"Optimized:              Sharpe {val_result.results['sharpe_ratio']:.2f}, PnL ${val_result.results['net_pnl_quote']:.2f}")
print(f"\nDelta: {val_result.results['sharpe_ratio'] - 1.29:+.2f} Sharpe")
print("\nIf delta > +0.3, inspect for overfitting before proceeding to OOS.")